# RNN, LSTM and BiLSTM: How the Data Actually Flows

This notebook has all the code from the two class notebooks (**RNN-Theory and Python Implementation** and **LSTM Theory and Python Implementation**), unchanged. In between, I've added explanation cells that answer one question again and again:

> *What exactly goes into this layer, what does it do with it, and what does it hand to the next layer?*

To make that concrete, I use one tiny toy example throughout: **2 sentences, 6 words in the vocabulary, sequence length 5**. Small enough that you can print every number and follow it with your eyes. Once the toy version makes sense, the real one (25,000 reviews, 20,000 words, length 100) is exactly the same thing, just bigger.

Cells marked **Toy demo** are new and run on their own. They don't need the dataset.

## Part 0: The journey of one sentence

Every model in this notebook follows the same road:

| Stage | What the data looks like | Shape (toy) | Shape (IMDB notebook) |
|---|---|---|---|
| Raw text | `"movie was great"` | 2 sentences | 2,773 reviews |
| Word to integer | `[1, 2, 5]` | list of lists (ragged) | list of lists |
| Pad / truncate | `[0, 0, 1, 2, 5]` | `(2, 5)` | `(2773, 100)` |
| Batching | a slice of rows | `(2, 5)` | `(32, 100)` |
| Embedding | each integer becomes a vector | `(2, 5, 4)` | `(32, 100, 128)` |
| Recurrent layer | reads the vectors one time step at a time | `(2, 3)` or `(2, 5, 3)` | `(32, 100)` or `(32, 100, 50)` |
| Dense + sigmoid | one probability per sentence | `(2, 1)` | `(32, 1)` |

Three words you'll see all the time:

**Batch** is how many sentences are processed together (first axis).
**Time steps / sequence length** is how many tokens each sentence has after padding (second axis).
**Features** is how many numbers describe one token (third axis). After an Embedding layer, that's the embedding size.

The rule for every recurrent layer in Keras: **input must be 3D, `(batch, timesteps, features)`**. Keep that in mind and most shape errors disappear.

In [1]:
# Toy demo: our tiny dataset
import numpy as np
np.set_printoptions(precision=3, suppress=True)

toy_vocab = {"movie": 1, "was": 2, "not": 3, "good": 4, "great": 5, "boring": 6}   # 0 is reserved for padding
toy_sentences = ["movie was great", "movie was not good"]

toy_ints = [[toy_vocab[w] for w in s.split()] for s in toy_sentences]
for s, ids in zip(toy_sentences, toy_ints):
    print(f"{s!r:25} -> {ids}")

'movie was great'         -> [1, 2, 5]
'movie was not good'      -> [1, 2, 3, 4]


Notice the two sentences have different lengths (3 and 4). A neural network needs a rectangle of numbers, so the next step is always **padding**. We'll see that inside the class code below.

---
# Part 1: Simple RNN (class notebook: *RNN-Theory and Python Implementation*)

Everything from here until Part 2 is the original notebook, cell for cell. Outputs have been cleared so the file stays light; run it top to bottom in Colab. My added cells are the ones starting with **Explanation** or **Toy demo**.

Note: the theory cells refer to images in the `img/` folder on Drive, so a few pictures may not show outside the original Colab setup.

# RNN (Recurrent Neural Network)

## 1)  After CNN, why RNN is there?

The backpropagation algorithm in CNN (convolutional neural network), we know that their output only considers the influence of the previous input and does not consider the influence of other moments of input, such as simple cats, dogs, handwritten numbers and other single objects.

However, for some related to time, such as the prediction of the next moment of the video, the prediction of the content of the previous and subsequent documents, etc., the performance of these algorithms is not satisfactory. Therefore, RNN should be applied and was born.

## 2) What is RNN?

RNN is a special neural network structure, which is proposed based on the view that " human cognition is based on past experience and memory ". It is different from DNN and CNN in that it not only considers the input of the previous moment, And it gives the network a 'memory' function of the previous content .

The reason why RNN is called recurrent neural network is that the current output of a sequence is also related to the previous output. The specific manifestation is that the network memorizes the previous information and applies it to the current output calculation, that is, the nodes between the hidden layers are connected, and the input of the hidden layer includes not only the output of the input layer It also includes the output of the hidden layer from the previous moment.


## 3) What are the main application areas of RNN ?

There are many application fields of RNN. It can be said that as long as the problem of chronological order is considered, RNN can be used to solve it. Here are some common application fields:

    ① Natural Language Processing (NLP) : There are video processing ,  text generation , language model , image processing

    ② Machine translation , machine writing novels

    ③ Speech recognition

    ④ Image description generation

    ⑤ Text similarity calculation

    ⑥ New application areas such as music recommendation , Netease koala product recommendation , Youtube video recommendation, etc.

### Different types of RNN

![alt](https://miro.medium.com/max/1400/0*1PKOwfxLIg_64TAO.jpeg)

#### One-to-one:

This also called as Plain/Vaniall Neural networks. It deals with Fixed size of input to Fixed size of Output where they are independent of previous information/output.

Ex: Image classification.


#### One-to-Many:

it deals with fixed size of information as input that gives sequence of data as output.

Ex:Image Captioning takes image as input and outputs a sentence of words.

![alt](https://miro.medium.com/max/1400/0*d9FisCKzVZ29SxUu.png)

#### Many-to-One:

It takes Sequence of information as input and ouputs a fixed size of output.

Ex:sentiment analysis where a given sentence is classified as expressing positive or negative sentiment.


#### Many-to-Many:

It takes a Sequence of information as input and process it recurrently outputs a Sequence of data.

Ex: Machine Translation, where an RNN reads a sentence in English and then outputs a sentence in French.

## RNN model structure

Earlier we said that RNN has the function of "memory" of time, so how does it realize the so-called "memory"?

<img src="/files/img/1.png" width="400">

<center>Figure 1 RNN structure diagram </center>

As shown in Figure 1, we can see that the RNN hierarchy is simpler than CNN.It mainly consists of an input layer , a Hidden Layer , and an output layer .

And you will find that **there is an arrow in the Hidden Layer to  indicate the cyclic update of the data.This is the method to implement the time memory function.**

![alt](https://miro.medium.com/max/1400/1*xn5kA92_J5KLaKcP7BMRLA.gif)

* t — time step
* X — input
* h — hidden state
* length of X — size/dimension of input
* length of h — no. of hidden units.

![alt](img/2.png)

<center>Figure 2 Unfolded RNN Diagram </center>

Figure 2 shows the hierarchical expansion of the Hidden Layer where

**T-1, t, t + 1 represent the time series**

**X represents the input sample**

**St represents the memory of the sample at time t, St = f (W * St -1 + U * Xt).**

**W is the weight of the input**

**U is the weight of the input sample at the moment**

**V is the weight of the output sample.**


### Feedforward

At t = 1, the general initialization input S0 = 0, randomly initializes W, U, V, and calculates the following formula:

![alt](img/21.png)

Among them, f and g are activation functions. Among them, f can be tanh, relu, sigmoid and other activation functions, g is usually softmax or other.

Time advances, and the state s1 at this time, as the memory state at time 1, will participate in the prediction activity at the next time, that is:

![alt](img/22.png)

By analogy, you can get the final output value:

![alt](img/23.png)


Note :

1. Here W, U, V are equal at each moment ( weight sharing ).

2. The hidden state can be understood as: S = f (existing input + past memory summary )


## Back propagation through TIME of RNN

Earlier we introduced the forward propagation method of RNN, so how are the weight parameters W, U, and V of the RNN updated?

Each output will have a value Ot error value , Et the total error may be expressed as: ![alt](img/31.png)

 The loss function can use either the cross-entropy loss function or the squared error loss function .

Because the output of each step does not only depend on the network of the current step, but also the state of the previous steps, then this modified BP algorithm is called Backpropagation Through Time ( BPTT ), which is the reverse transfer of the error value at the output end. The gradient descent method is updated.

That is, the gradient of the parameter is required:

![alt](img/32.png)

 First we solve the update method of W. From the previous update of W, it can be seen that it is the sum of the partial derivatives of the deviations at each moment.

Here we take time t = 3 as an example.According to the chain derivation rule, we can get the partial derivative at time t = 3 as:


![alt](img/331.png)


At this time, according to the formula, ![alt](img/331.png) we will find that in addition to W, S3 is also related to S2 at the previous moment.

![alt](https://miro.medium.com/max/1400/0*ENwCVS8XI8cjCy55.jpg)

For S3, expand directly to get the following formula:

![alt](img/34.png)

For S2, expand directly to get the following formula:

![alt](img/35.png)

For S1, expand directly to get the following formula:

![alt](img/36.png)

Combine the above three formulas to get:

![alt](img/37.png)

This gives the formula:

![alt](img/381.png)





What is to be explained here is ![alt](img/39.png)  that it means that S3 directly differentiates W without considering the effect of S2 (that is, for example, y = f (x) * g (x) is the same as the derivative of x)

The second is the update method for U. Since the parameter U and W are similar, they will not be described here, and the specific formula finally obtained is as follows:

![alt](img/40.png)

Finally, give the updated formula of V (V is only related to output O):

![alt](img/41.png)




### Going little deeper

Let’s focus on one error term et.

You’ve calculated the cost function et, and now you want to propagate your cost function back through the network because you need to update the weights.

Essentially, every single neuron that participated in the calculation of the output, associated with this cost function, should have its weight updated in order to minimize that error. And the thing with RNNs is that it’s not just the neurons directly below this output layer that contributed but all of the neurons far back in time. So, you have to propagate all the way back through time to these neurons.

The problem relates to updating wrec (weight recurring) – the weight that is used to connect the hidden layers to themselves in the unrolled temporal loop.

For instance, to get from xt-3 to xt-2 we multiply xt-3 by wrec. Then, to get from xt-2 to xt-1 we again multiply xt-2 by wrec. So, we multiply with the same exact weight multiple times, and this is where the problem arises: when you multiply something by a small number, your value decreases very quickly.

As we know, weights are assigned at the start of the neural network with the random values, which are close to zero, and from there the network trains them up. But, when you start with wrec close to zero and multiply xt, xt-1, xt-2, xt-3, … by this value, your gradient becomes less and less with each multiplication.

#### What does this mean for the network?

The lower the gradient is, the harder it is for the network to update the weights and the longer it takes to get to the final result.

For instance, 1000 epochs might be enough to get the final weight for the time point t, but insufficient for training the weights for the time point t-3 due to a very low gradient at this point. However, the problem is not only that half of the network is not trained properly.

The output of the earlier layers is used as the input for the further layers. Thus, the training for the time point t is happening all along based on inputs that are coming from untrained layers. So, because of the vanishing gradient, the whole network is not being trained properly.

To sum up, if wrec is small, you have vanishing gradient problem, and if wrec is large, you have exploding gradient problem.

For the vanishing gradient problem, the further you go through the network, the lower your gradient is and the harder it is to train the weights, which has a domino effect on all of the further weights throughout the network.

That was the main roadblock to using Recurrent Neural Networks. But let’s now check what are the possible solutions to this problem.

Solutions to the vanishing gradient problem
In case of exploding gradient, you can:

*    stop backpropagating after a certain point, which is usually not optimal because not all of the weights get updated;
*    penalize or artificially reduce gradient;
*    put a maximum limit on a gradient.


In case of vanishing gradient, you can:

*    initialize weights so that the potential for vanishing gradient is minimized;
*    have Echo State Networks that are designed to solve the vanishing gradient problem;
*    have Long Short-Term Memory Networks (LSTMs).




### Advantages of Recurrent Neural Network

-    RNN can model sequence of data so that each sample can be assumed to be dependent on previous ones
-    Recurrent neural network are even used with convolutional layers to extend the effective pixel neighbourhood.

### Disadvantages of Recurrent Neural Network

-   Gradient vanishing and exploding problems.
-   Training an RNN is a very difficult task.
-   It cannot process very long sequences if using tanh or relu as an activation function.

# Building Classifers using the Reuters Dataset

In [ ]:
import numpy as np
import tensorflow as tf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
with open('/content/drive/MyDrive/Artificial_Intelligence_Engineering/nlp/RNN and LSTM/reviews.txt', 'r') as f:
    reviews = f.read()
with open('/content/drive/MyDrive/Artificial_Intelligence_Engineering/nlp/RNN and LSTM/labels.txt', 'r') as f:
    labels = f.read()

In [ ]:
reviews

In [ ]:
labels

In [ ]:
list_reviews = reviews.split("\n")
labels_list = labels.split("\n")

In [ ]:
print(type(list_reviews))
print(len(list_reviews))
print(len(labels_list))

In [ ]:
list_reviews[0]

In [ ]:
labels_list[0]

In [ ]:
from string import punctuation
punctuation

In [ ]:
from string import punctuation
clean_list_reviews = []
for doc in list_reviews:
  mydoc = ""
  for j in doc:
    if j not in punctuation:
      mydoc += j
  clean_list_reviews.append(mydoc)

In [ ]:
clean_list_reviews[0]

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt') # punkt is needed for word tokenization
from nltk.corpus import stopwords
stop_words = list(set(stopwords.words('english')))
print(stop_words)

In [ ]:
cleaner_list_reviews = []
for sen in clean_list_reviews:
  mysen = []
  for word in sen.split():
    if word.lower() in stop_words:
      pass
    else:
      mysen.append(word.lower())
  mynewsen = " ".join(mysen)
  cleaner_list_reviews.append(mynewsen)

In [ ]:
len(cleaner_list_reviews)

## Data preprocessing

The first step when building a neural network model is getting your data into the proper form to feed into the network. Since we're using embedding layers, we'll need to encode each word with an integer. We'll also want to clean it up a bit.

You can see an example of the reviews data above. We'll want to get rid of those periods. Also, you might notice that the reviews are delimited with newlines `\n`. To deal with those, I'm going to split the text into each review using `\n` as the delimiter. Then I can combined all the reviews back together into one big string.

First, let's remove all punctuation. Then get all the text without the newlines and split it into individual words.

In [ ]:
words = []
for sen in  cleaner_list_reviews:
  for word in sen.split():
    words.append(word.lower())

In [ ]:
len(words) # Total number of words used in corpus

In [ ]:
len(set(words)) # Total number of unique words used in corpus

### Encoding the words

The embedding lookup requires that we pass in integers to our network. The easiest way to do this is to create dictionaries that map the words in the vocabulary to integers. Then we can convert each of our reviews into integers so they can be passed into the network.

Now you're going to encode the words with integers. Build a dictionary that maps words to integers. Later we're going to pad our input vectors with zeros, so make sure the integers **start at 1, not 0**.

Also, convert the reviews to integers and store the reviews in a new list called `reviews_ints`.

In [ ]:
from collections import Counter
counts = Counter(words)

In [ ]:
counts

In [ ]:
for i,j in enumerate(['ab','bc','cd'],1):
  print(i,j)

In [ ]:
mydict = {'a':4,'b':9,'c':2}


In [ ]:
sorted(mydict)

In [ ]:
print(mydict.values())
print(mydict.keys())
print(list(zip(mydict.values(),mydict.keys())))

In [ ]:
list(zip(mydict.values(),mydict.keys()))

In [ ]:
abc = {i[1]:i[0] for i in sorted(list(zip(mydict.values(),mydict.keys())),reverse=True)}
abc

In [ ]:
xyz = {}
for i,j in enumerate(abc.keys(),1):
  xyz[j] = i

In [ ]:
xyz

In [ ]:
sorted_dict = {i[1]:i[0] for i in sorted(list(zip(counts.values(),counts.keys())),reverse=True)}

In [ ]:
sorted_dict

In [ ]:
dict1 = {"a":1,'b':3,'c':2}
# b:3,c:2,a:1

{i[1]:i[0] for i in sorted(list(zip(dict1.values(),dict1.keys())),reverse=True)}

In [ ]:
sorted_dict = {k: v for k, v in sorted(counts.items(), key=lambda item: item[1],reverse=True)}

In [ ]:
print(sorted_dict)

In [ ]:
myvocab = list(sorted_dict.keys())
print(myvocab)
print(len(myvocab))

In [ ]:
type(clean_list_reviews)

In [ ]:
clean_list_reviews[0]

In [ ]:
clean_list_reviews[1]

In [ ]:
vocab_to_int = {word: i for i, word in enumerate(myvocab, 1)}
print(vocab_to_int)

In [ ]:
cleaner_list_reviews[1]

In [ ]:
# Create your dictionary that maps vocab words to integers here # start from 1

# Convert the reviews to integers, same shape as reviews list, but with integers
review_ints = []
for each in cleaner_list_reviews:
    review_ints.append([vocab_to_int[word] for word in each.split()])

In [ ]:
len(review_ints)

In [ ]:
print(review_ints[0])

In [ ]:
print(type(review_ints))
print(len(review_ints))
print(type(review_ints[0]))
print(len(review_ints[0]))
print(review_ints[0])
print(len(review_ints[1]))
print(len(review_ints[2]))

### Explanation: Checkpoint, what `review_ints` looks like now

At this point each review is a **Python list of integers**, one integer per word, and every list has a different length. This is exactly our toy:

```
"movie was great"     -> [1, 2, 5]
"movie was not good"  -> [1, 2, 3, 4]
```

Why integers start from 1: we are going to fill empty slots with **0** during padding, so 0 must not mean any real word.

A neural network can't take a ragged list like this. We need every review to be the **same length**. That's the next section.

### Encoding the labels

Our labels are "positive" or "negative". To use these labels in our network, we need to convert them to 0 and 1.

Convert labels from `positive` and `negative` to 1 and 0, respectively.

In [ ]:
labels

In [ ]:
import numpy as np
labels = [1 if label == 'positive' else 0 for label in labels_list]
labels = np.array(labels)

In [ ]:
labels

If you built `labels` correctly, you should see the next output.

In [ ]:
[i for i,j in enumerate([len(lst) for lst in review_ints])if j == 0]

In [ ]:
lst = [4,5,2,0,3,1,5,0,2,0,1,3]
[i for i,j in enumerate(lst) if j == 0]

In [ ]:
review_ints[25000]

In [ ]:
len_of_list = []
for lst in review_ints:
  len_of_list.append(len(lst))

In [ ]:
print(len_of_list)

In [ ]:
np.min(len_of_list)

In [ ]:
np.where(np.array(len_of_list) == 0)

In [ ]:
review_ints[25000]

In [ ]:
labels[25000]

In [ ]:
review_ints[25000]

In [ ]:
review_ints = review_ints[:-1]
labels = labels[:-1]

In [ ]:
print(len(review_ints))
print(len(labels))

In [ ]:
print(len(review_ints[1]))
print(len(review_ints[2]))
print(len(review_ints[3]))

In [ ]:
np.array([len(lst) for lst in review_ints]).max()

Okay, a couple issues here. We seem to have one review with zero length. And, the maximum review length is way too many steps for our RNN. Let's truncate to 200 steps. For reviews shorter than 200, we'll pad with 0s. For reviews longer than 200, we can truncate them to the first 200 characters.

First, remove the review with zero length from the `reviews_ints` list.

In [ ]:
# Filter out that review with 0 length
# review_ints = [review for review in review_ints if (len(review) > 0)]

In [ ]:
len(review_ints)

### Explanation: Padding and truncating, step by step

After padding, the reviews become a 2D matrix of shape `(number_of_reviews, seq_len)`. Each row is one review and each column is one **time step**.

Two decisions matter a lot for RNNs:

**Where do the zeros go?** `padding='pre'` puts them at the start, `padding='post'` at the end. An RNN reads left to right and, with `return_sequences=False`, we only keep its **last** output. With post-padding, the last several steps are all zeros, so the useful signal from the actual words has to survive a long stretch of "nothing" before we read the answer. Pre-padding puts the real words right before the output, which usually trains better.

**When a review is too long, which part is kept?** Keras `pad_sequences` cuts from the start by default (`truncating='pre'`) and keeps the **last** `maxlen` words. The manual loop in this notebook does `review[:seq_len]` and keeps the **first** 200. Both are valid; just be aware they are different.

Run the toy demo to see all of this on 2 sentences.

In [2]:
# Toy demo: padding and truncating
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("Original      :", toy_ints)
print("\npadding='pre' (default), maxlen=5:\n", pad_sequences(toy_ints, maxlen=5))
print("\npadding='post', maxlen=5:\n", pad_sequences(toy_ints, maxlen=5, padding='post'))
print("\nmaxlen=3 -> long sentences get cut. Default keeps the LAST 3 words:\n", pad_sequences(toy_ints, maxlen=3))
print("\ntruncating='post' keeps the FIRST 3 words:\n", pad_sequences(toy_ints, maxlen=3, truncating='post'))

Original      : [[1, 2, 5], [1, 2, 3, 4]]

padding='pre' (default), maxlen=5:
 [[0 0 1 2 5]
 [0 1 2 3 4]]

padding='post', maxlen=5:
 [[1 2 5 0 0]
 [1 2 3 4 0]]

maxlen=3 -> long sentences get cut. Default keeps the LAST 3 words:
 [[1 2 5]
 [2 3 4]]

truncating='post' keeps the FIRST 3 words:
 [[1 2 5]
 [1 2 3]]


**Teaching note on the next few cells:** cell `features = pad_sequences(review_ints, padding='post')` pads every review to the longest one (1,442 words, post-padded). The loop after it rebuilds `features` with length 200 and pre-padding, overwriting the first result. The model cells later reshape to `(20000, 1442, 1)`, so they expect the **first** version. If you run everything top to bottom, the reshape will fail. Run one of the two padding cells, not both, and make the reshape match (`1442` or `200`).

Now, create an array `features` that contains the data we'll pass to the network. The data should come from `review_ints`, since we want to feed integers to the network. Each row should be 200 elements long. For reviews shorter than 200 words, left pad with 0s. That is, if the review is `['best', 'movie', 'ever']`, `[117, 18, 128]` as integers, the row will look like `[0, 0, 0, ..., 0, 117, 18, 128]`. For reviews longer than 200, use on the first 200 words as the feature vector.

This isn't trivial and there are a bunch of ways to do this. But, if you're going to be building your own deep learning networks, you're going to have to get used to preparing your data.



In [ ]:
review_ints[1]

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
features = pad_sequences(review_ints,padding='post')

In [ ]:
features.shape

In [ ]:
seq_len = 200
features = []
for review in review_ints:
    review_len = len(review)
    len_diff = seq_len - review_len
    if len_diff <= 0:
        features.append(review[:seq_len])

    else:
        padding = [0] * len_diff
        padded_feature = padding + review
        features.append(padded_feature)

features = np.asarray(features)

In [ ]:
print(len(features))
print(len(features[0]))
print(len(features[1]))
print(len(features[2]))
print(features.shape)
print(labels.shape)

## Training, Validation, Test



With our data in nice shape, we'll split it into training, validation, and test sets.

Create the training, validation, and test sets here. You'll need to create sets for the features and the labels, `train_x` and `train_y` for example. Define a split fraction, `split_frac` as the fraction of data to keep in the training set. Usually this is set to 0.8 or 0.9. The rest of the data will be split in half to create the validation and testing data.

In [ ]:
split_frac = 0.8
split_idx = int(len(features) * split_frac)

train_x, val_x = features[:split_idx], features[split_idx:]
train_y, val_y = labels[:split_idx], labels[split_idx:]

test_idx = int(len(val_x) * 0.5)
val_x, test_x = val_x[:test_idx], val_x[test_idx:]
val_y, test_y = val_y[:test_idx], val_y[test_idx:]

print("\t\t\tFeature Shapes:")
print("Train set: \t\t{},{}".format(train_x.shape,train_y.shape),
      "\nValidation set: \t{},{}".format(val_x.shape,val_y.shape),
      "\nTest set: \t\t{},{}".format(test_x.shape,test_y.shape))

In [ ]:
np.unique(train_y)

### Explanation: Splitting gives us our batch axis

After the split we have `train_x` of shape `(20000, L)` where `L` is the padded length. Row = one review, column = one time step. When we call `model.fit(..., batch_size=50)`, Keras slices this into chunks of 50 rows. So inside the network, one forward pass sees an input of shape `(50, L, ...)`.

For 20,000 reviews and batch size 50, one epoch = 20000 / 50 = **400 steps** (weight updates).

## Simple RNN

In [ ]:
pip install scikeras

### Explanation: What the Reuters experiment feeds into `SimpleRNN`

Reading the cell below in shapes:

1. `reuters.load_data(..., maxlen=50)` keeps only news items shorter than 50 words.
2. `pad_sequences(..., padding='post')` makes every item the same length. Here that length turns out to be **49**.
3. `.reshape((N, 49, 1))` turns 2D into 3D. That last `1` means **each time step has exactly one feature: the raw word ID**.
4. `SimpleRNN(50, input_shape=(49,1), return_sequences=False)` reads 49 steps, 1 number per step, and returns only the final hidden state, shape `(batch, 50)`.
5. `Dense(46)` + softmax gives 46 probabilities, one per news topic.

**Heads up for today's Keras:** `optimizers.Adam(lr=0.001)` now throws an error. Use `optimizers.Adam(learning_rate=0.001)`.

In [ ]:
import numpy as np

from sklearn.metrics import accuracy_score
from keras.datasets import reuters
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, SimpleRNN, Activation
from keras import optimizers
# from keras.wrappers.scikit_learn import KerasClassifier
# from keras.wrappers import KerasClassifier

# parameters for data load
num_words = 30000
maxlen = 50
test_split = 0.3

(X_train, y_train), (X_test, y_test) = reuters.load_data(num_words = num_words, maxlen = maxlen, test_split = test_split)



# pad the sequences with zeros
# padding parameter is set to 'post' => 0's are appended to end of sequences
X_train = pad_sequences(X_train, padding = 'post')
X_test = pad_sequences(X_test, padding = 'post')

X_train = np.array(X_train).reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = np.array(X_test).reshape((X_test.shape[0], X_test.shape[1], 1))

y_data = np.concatenate((y_train, y_test))
y_data = to_categorical(y_data)
y_train = y_data[:1395]
y_test = y_data[1395:]


model = Sequential()
model.add(SimpleRNN(50, input_shape = (49,1), return_sequences = False))
model.add(Dense(46))
model.add(Activation('softmax'))

adam = optimizers.Adam(lr = 0.001)
model.compile(loss = 'categorical_crossentropy', optimizer = adam, metrics = ['accuracy'])



# model = KerasClassifier(build_fn = vanilla_rnn, epochs = 200, batch_size = 50, verbose = 1)
model.fit(X_train, y_train,epochs = 200,batch_size = 50, verbose = 1)

# y_pred = model.predict(X_test)
# y_test_ = np.argmax(y_test, axis = 1)

# print(accuracy_score(y_pred, y_test_))

### Explanation: Why reshape `(20000, 1442)` into `(20000, 1442, 1)`?

Recurrent layers insist on 3D input: `(batch, timesteps, features)`. We have no Embedding layer in this model, so each word is represented by **one number** (its ID). That's `features = 1`.

```
before : (20000, 1442)      20000 reviews x 1442 word IDs
after  : (20000, 1442, 1)   20000 reviews x 1442 time steps x 1 feature per step
```

This works technically, but it's worth pausing on with students: feeding the raw ID as a number tells the network that word #900 is "bigger" than word #3, which means nothing. That's the limitation that the **Embedding layer** in Part 2 fixes: each word gets its own learned vector instead of a single arbitrary number.

In [ ]:
train_x.shape

In [ ]:
train_x = train_x.reshape(20000,1442,1)

In [ ]:
train_x.shape

## Explanation: Inside a `SimpleRNN` layer, one time step at a time

This is the heart of it. A SimpleRNN layer with `units = 3` keeps a **hidden state** `h`, a vector of 3 numbers, its "memory". It starts at all zeros. Then, for every time step `t`:

$$h_t = \tanh(x_t \cdot W_x + h_{t-1} \cdot W_h + b)$$

| Symbol | Meaning | Toy shape |
|---|---|---|
| $x_t$ | the vector for the token at step t | `(4,)` |
| $W_x$ | input weights (`kernel`) | `(4, 3)` |
| $h_{t-1}$ | memory from the previous step | `(3,)` |
| $W_h$ | memory weights (`recurrent_kernel`) | `(3, 3)` |
| $b$ | bias | `(3,)` |
| $h_t$ | new memory = output at this step | `(3,)` |

The **same** $W_x$, $W_h$, $b$ are used at every step. That's why the number of parameters doesn't depend on sentence length.

Let's prove it. The demo builds a real Keras `SimpleRNN`, takes its weights out, repeats the calculation with a plain `for` loop in NumPy, and checks that the numbers match.

In [ ]:
# Toy demo: step 1, make the 3D input (batch, timesteps, features)
import tensorflow as tf
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Bidirectional

tf.random.set_seed(7)
toy_batch = pad_sequences(toy_ints, maxlen=5)        # (2, 5): 2 sentences x 5 time steps
toy_emb   = Embedding(input_dim=7, output_dim=4)(toy_batch).numpy()   # (2, 5, 4)

print("toy_batch (word IDs):\n", toy_batch)
print("\ntoy_emb shape:", toy_emb.shape, " -> (batch=2, timesteps=5, features=4)")
print("\nSentence 0, all 5 time steps (each row = one token's vector):\n", toy_emb[0])

In [ ]:
# Toy demo: step 2, SimpleRNN by hand vs Keras
rnn_layer = SimpleRNN(3, return_sequences=True)
keras_out = rnn_layer(toy_emb).numpy()               # (2, 5, 3)

Wx, Wh, b = rnn_layer.get_weights()
print("Wx:", Wx.shape, " Wh:", Wh.shape, " b:", b.shape)

x = toy_emb[0]                                       # sentence 0 -> (5, 4)
h = np.zeros(3)                                      # memory starts empty
print("\nReading 'movie was great' (pre-padded) one token at a time:")
for t in range(5):
    h = np.tanh(x[t] @ Wx + h @ Wh + b)
    print(f"  t={t}  word_id={toy_batch[0, t]}  ->  h_t = {h}")

print("\nKeras output for sentence 0:\n", keras_out[0])
print("\nDo my loop and Keras agree?", np.allclose(h, keras_out[0, -1], atol=1e-5))

Things to point out while this runs:

**The two padding steps (word_id 0) still change `h`.** The zero-ID gets a vector from the embedding table like any other word, and without masking the RNN processes it. (`Embedding(..., mask_zero=True)` tells later layers to skip them.)

**Each `h_t` depends on everything before it.** `h` at t=4 has "seen" `movie`, `was`, `great` in order. That's the memory.

**Your exact numbers will differ from your neighbour's** because the weights are random, but the last line should say `True`.

### Explanation: The whole batch moves together

In the loop above I processed one sentence. In reality, at time step `t` the layer takes **column t of every sentence in the batch** at once and does a single matrix multiplication. Time steps are sequential; sentences in a batch are parallel.

In [ ]:
# Toy demo: step 3, same calculation, whole batch at once
H = np.zeros((2, 3))                                  # one memory vector per sentence
for t in range(5):
    x_t = toy_emb[:, t, :]                            # token t of BOTH sentences -> (2, 4)
    H = np.tanh(x_t @ Wx + H @ Wh + b)                # (2,4)@(4,3) + (2,3)@(3,3) -> (2,3)
    print(f"t={t}: x_t {x_t.shape} -> H {H.shape}")
print("\nMatches Keras for both sentences?", np.allclose(H, keras_out[:, -1, :], atol=1e-5))

### Explanation: `return_sequences=True` vs `False`, and why stacking needs it

| Setting | What the layer returns | Toy shape | Use it when |
|---|---|---|---|
| `return_sequences=True` | $h_t$ at **every** step | `(2, 5, 3)` | the next layer is another RNN/LSTM (it needs a sequence) |
| `return_sequences=False` (default) | only the **last** $h_t$ | `(2, 3)` | the next layer is Dense (one answer per sentence) |

That's why the model in the next code cell stacks three SimpleRNNs as `True, True, False`: the first two pass a full sequence upward, the last one squeezes the sentence into a single summary vector.

**Parameter count for SimpleRNN** = `units x (input_features + units + 1)`

For the model below:
first SimpleRNN, input features = 1: `50 x (1 + 50 + 1) = 2,600`
second and third SimpleRNN, input features = 50: `50 x (50 + 50 + 1) = 5,050` each

The next demo runs dummy data through the same architecture layer by layer, so you can watch the shape change at each stop.

In [ ]:
# Toy demo: step 4, shape walk through the class model (dummy data, 4 reviews)
from tensorflow.keras.layers import Dense, Flatten
dummy = np.random.randint(0, 5000, size=(4, 1442, 1)).astype("float32")   # 4 reviews, 1442 steps, 1 feature

walk = [SimpleRNN(50, return_sequences=True), SimpleRNN(50, return_sequences=True),
        SimpleRNN(50, return_sequences=False), Flatten(), Dense(46), Dense(46), Dense(46),
        Dense(1, activation="sigmoid")]
out = dummy
print(f"{'input':38} {str(out.shape):16}")
for layer in walk:
    out = layer(out)
    desc = layer.__class__.__name__ + (f"(return_sequences={layer.return_sequences})" if hasattr(layer, "return_sequences") else "")
    print(f"{desc:38} {str(tuple(out.shape)):16} params={layer.count_params():,}")

Now the original model cell. With the walk above in mind, each line should read like a sentence: "3D in, sequence out, sequence out, summary out, a few Dense layers, one probability."

In [ ]:
import numpy as np
import tensorflow as tf

from sklearn.metrics import accuracy_score

# from keras.preprocessing.sequence import pad_sequences
from keras.layers import Dense, Dropout, Embedding, LSTM, Bidirectional,SimpleRNN, Activation,Flatten
#from keras.utils import to_categorical
from keras.models import Sequential

from keras import optimizers
# from keras.wrappers.scikit_learn import KerasClassifier


model = Sequential()
model.add(SimpleRNN(50, input_shape = (1442,1), return_sequences = True))
model.add(SimpleRNN(50,return_sequences = True))
model.add(SimpleRNN(50,return_sequences = False))
# model.add(LSTM(64))
model.add(Flatten())
model.add(Dense(46))
model.add(Dense(46))
model.add(Dense(46))
model.add(Dense(1,activation = "sigmoid"))


adam = tf.keras.optimizers.Adam(learning_rate = 0.001)
model.compile(loss = 'binary_crossentropy', optimizer = adam, metrics = ['accuracy'])

model.fit(train_x, train_y, epochs = 1, batch_size = 50, verbose = 1)




In [ ]:
model.summary()

### Explanation: Getting predictions back

`model.predict(...)` returns **probabilities** of shape `(num_reviews, 1)`, e.g. `0.73`. To compare with 0/1 labels, turn them into classes first.

A few things to fix when students run the next cells in order:

1. `y_pred` is used in the cells right below before it's created. Run the `model.predict` cell first.
2. `test_x` must get the same 3D reshape as `train_x`: `test_x.reshape(-1, 1442, 1)`.
3. `accuracy_score` needs class labels, not probabilities.

A working version:

```python
y_prob = model.predict(test_x.reshape(-1, 1442, 1))
y_pred = (y_prob > 0.5).astype(int).ravel()
print(accuracy_score(test_y, y_pred))
```

In [ ]:
len(test_y)

In [ ]:
y_pred.shape

In [ ]:
test_y.shape

In [ ]:
y_pred

In [ ]:
y_pred = model.predict(test_x)
#y_test_ = np.argmax(test_y,)

print(accuracy_score(y_pred, test_y))

---
# Part 2: LSTM and BiLSTM (class notebook: *LSTM Theory and Python Implementation*)

All the code from the LSTM notebook, unchanged, in the original order. The long theory write-up on gates (with diagrams from the `image/` folder) stays in the original notebook; the cells here focus on what happens to the data.

The big upgrade over Part 1: an **Embedding layer**. Instead of feeding one raw number per word, every word gets its own learned vector of 128 numbers.

In [ ]:
# Toy demo setup: recreate the toy objects so Part 2 runs without Part 1
import numpy as np, tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional
np.set_printoptions(precision=3, suppress=True)

toy_vocab = {"movie": 1, "was": 2, "not": 3, "good": 4, "great": 5, "boring": 6}
toy_ints  = [[toy_vocab[w] for w in s.split()] for s in ["movie was great", "movie was not good"]]
toy_batch = pad_sequences(toy_ints, maxlen=5)                                  # (2, 5)
tf.random.set_seed(7)
toy_emb   = Embedding(input_dim=7, output_dim=4)(toy_batch).numpy()           # (2, 5, 4)
print(toy_batch.shape, "->", toy_emb.shape)

### Explanation: What `imdb.load_data` gives us

It returns reviews that are **already converted to integers** (the same thing we did by hand in Part 1).

`num_words=20000` keeps only the 20,000 most frequent words; rarer words are replaced by an "unknown" ID.
`maxlen=100` in `load_data` **drops** every review longer than 100 words. That's why this cell gives only about 2,700 training reviews instead of 25,000. (The BiLSTM cell at the end doesn't pass `maxlen` to `load_data`, so it keeps all 25,000 and truncates them later.)

The next cell prints lengths: they're all different, just like our toy `[1, 2, 5]` and `[1, 2, 3, 4]`.

In [ ]:
#importing libraries
# from __future__ import print_function
import numpy as np

from tensorflow.keras.preprocessing import sequence
from keras.models import Sequential
from keras.layers import Dense, Dropout, Embedding, LSTM, Bidirectional,SimpleRNN
from keras.datasets import imdb


max_features = 20000
# cut texts after this number of words
# (among top max_features most common words)
maxlen = 100
# batch_size = 32

print('Loading data...')
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features,maxlen=maxlen)
print(len(x_train), 'train sequences')
print(len(x_test), 'test sequences')

In [ ]:
print(type(x_train))
print(x_train.shape)
print(type(x_train[0]))
print(type(x_train[1]))
print(len(x_train[0]))
print(len(x_train[1]))
print(len(x_train[3]))
print(len(x_train[4]))
print(len(x_train[5]))
print(len(x_train[6]))
print(len(x_train[2772]))

### Explanation: Padding to 100

`x_train` is a 1D array whose items are lists, so `x_train.shape` shows `(2773,)`, not a matrix yet. After `pad_sequences(..., maxlen=100)` it becomes a proper 2D matrix `(2773, 100)`: 2,773 reviews, 100 time steps each. Default is **pre**-padding, so zeros come first and the real words sit at the end, right before the LSTM gives its final answer. Look at `x_train[2]` to see the zeros at the start.

In [ ]:
x_train = sequence.pad_sequences(x_train, maxlen=100)
x_test = sequence.pad_sequences(x_test, maxlen=100)
print(len(x_train[0]))
print(len(x_train[1]))
print(len(x_train[2]))
print(len(x_train[3]))
print(len(x_train[4]))

In [ ]:
x_train[2]

In [ ]:
# 2773 X 100 X 20000
x_train.shape

## Explanation: What the Embedding layer really does

The comment in the next cell describes the embedding as a matrix multiplication: each word as a one-hot vector of length 20,000, times a `(20000 x 128)` table, giving `(100 x 128)` per sentence. That's the right **mental model**, and the math gives exactly that answer.

In practice Keras never builds the giant one-hot matrix. It does a **lookup**: word ID 5 means "fetch row 5 of the table". Multiplying a one-hot vector by a matrix picks out one row, so the result is identical; the lookup is just far cheaper. So `x_train` really is `(2773, 100)` of integers, and the `x 20000` exists only in the explanation.

```
one sentence  : (100,)        100 word IDs
after lookup  : (100, 128)    100 vectors of 128 numbers
one batch     : (32, 100) -> (32, 100, 128)
```

The table has `20000 x 128 = 2,560,000` numbers, and all of them are learned during training. That's where most of this model's parameters live.

The toy demo proves "one-hot x matrix" and "lookup" give the same thing.

In [ ]:
# Toy demo: Embedding = table lookup (same result as one-hot x matrix)
emb_layer = Embedding(input_dim=7, output_dim=4)     # vocab of 7 IDs (0-6), 4 numbers per word
looked_up = emb_layer(toy_batch).numpy()             # (2, 5) -> (2, 5, 4)
E = emb_layer.get_weights()[0]                       # the table itself: (7, 4)

print("Embedding table E, one row per word ID:\n", E)
print("\nSentence 0 IDs:", toy_batch[0])
print("Its vectors (rows of E picked by ID):\n", looked_up[0])

one_hot = np.eye(7)[toy_batch[0]]                    # (5, 7) one-hot version of sentence 0
print("\none_hot (5x7) @ E (7x4) equals the lookup?", np.allclose(one_hot @ E, looked_up[0]))

In [ ]:
# 2773 X 100 --> Batch X Sequence
# 2773 X 100 X 20000 --> Batch X Sequence X features
# 1 sentence -- > 100 X 20000
# Embedding layer --> 20000 X 128
# So when a single sentence goes inside embedding layer then
# (100 X 20000) (20000 X 128) --> (100 X 128)
# (100 X 128) is for one sentence
# For complete batch the o/p --> batch X 100 X 128

## Explanation: Inside an LSTM layer, one time step at a time

A SimpleRNN has one memory vector `h`. An LSTM has **two**:

**`c` (cell state)** is the long-term memory, the "conveyor belt" from the theory section.
**`h` (hidden state)** is the short-term memory, and also what the layer outputs.

At every time step the LSTM looks at the current word $x_t$ and the previous $h_{t-1}$, and computes four things:

| Name | Formula | Range | Job |
|---|---|---|---|
| forget gate $f$ | $\sigma(x_t W_f + h_{t-1} U_f + b_f)$ | 0 to 1 | how much of old `c` to keep |
| input gate $i$ | $\sigma(x_t W_i + h_{t-1} U_i + b_i)$ | 0 to 1 | how much new info to write |
| candidate $g$ | $\tanh(x_t W_g + h_{t-1} U_g + b_g)$ | -1 to 1 | the new info itself |
| output gate $o$ | $\sigma(x_t W_o + h_{t-1} U_o + b_o)$ | 0 to 1 | how much of `c` to reveal as `h` |

Then the two updates:

$$c_t = f \odot c_{t-1} + i \odot g \qquad\qquad h_t = o \odot \tanh(c_t)$$

($\odot$ = element-by-element multiply.)

**Why 4 x the parameters of a SimpleRNN?** Each of the four rows in the table has its own weights. Keras stores them side by side in one big matrix: kernel `(input_features, 4 x units)`, in the order **i, f, g, o**.

**LSTM parameter count** = `4 x units x (input_features + units + 1)`

In the class model below, `LSTM(100)` sits on top of `SimpleRNN(50)`, so input features = 50: `4 x 100 x (50 + 100 + 1) = 60,400`.

The demo does the gate maths by hand in NumPy and checks it against Keras.

In [ ]:
# Toy demo: LSTM by hand vs Keras
tf.random.set_seed(7)
lstm_layer = LSTM(3, return_sequences=True, return_state=True)
seq_out, h_last, c_last = lstm_layer(toy_emb)        # toy_emb is (2, 5, 4) from Part 1
K, U, bias = lstm_layer.get_weights()                # K:(4,12)  U:(3,12)  bias:(12,)
print("kernel:", K.shape, " recurrent_kernel:", U.shape, " bias:", bias.shape, "  (12 = 4 gates x 3 units)")

def sigmoid(z): return 1 / (1 + np.exp(-z))
n = 3
x = toy_emb[0]
h, c = np.zeros(n), np.zeros(n)
for t in range(5):
    z = x[t] @ K + h @ U + bias                      # all 4 gates in one go -> (12,)
    i, f, g, o = sigmoid(z[:n]), sigmoid(z[n:2*n]), np.tanh(z[2*n:3*n]), sigmoid(z[3*n:])
    c = f * c + i * g                                # forget some old memory, write some new
    h = o * np.tanh(c)                               # decide what to show
    print(f"t={t} id={toy_batch[0,t]}  forget={f}  input={i}  c={c}  h={h}")

print("\nKeras final h:", h_last.numpy()[0], "  mine:", h)
print("Keras final c:", c_last.numpy()[0], "  mine:", c)
print("Match?", np.allclose(h, h_last.numpy()[0], atol=1e-5) and np.allclose(c, c_last.numpy()[0], atol=1e-5))

What to discuss with students here:

**`c` changes gently.** The update is mostly *multiply by a number between 0 and 1, then add*. There's no squashing `tanh` on the conveyor belt itself, so information (and gradients during training) can travel many steps without fading. That's the whole reason LSTMs handle long sentences better than SimpleRNN.

**Read the gates like dials.** A forget value near 1 means "keep this memory", near 0 means "wipe it". Before training, the input and output gates hover around 0.5, but the forget gate sits near **0.73**. That's deliberate: Keras starts the forget-gate bias at 1 (`unit_forget_bias=True`), and sigmoid(1) = 0.73, so a fresh LSTM leans towards remembering. Training is what turns all of these into meaningful dials.

**`return_state=True`** is only for this demo, so we can see `c`. Normally you don't need it.

### Explanation: Shape walk through the class model

```
input                      (32, 100)          32 reviews x 100 word IDs
Embedding(20000,128)       (32, 100, 128)     each ID -> 128 numbers           2,560,000 params
SimpleRNN(50, seq=True)    (32, 100, 50)      memory at every step             50 x (128+50+1) = 8,950
LSTM(100)                  (32, 100)          only the LAST h (seq=False)      4 x 100 x (50+100+1) = 60,400
Dropout(0.5)               (32, 100)          randomly zeroes half, training only    0
Dense(1, sigmoid)          (32, 1)            one probability per review       100 + 1 = 101
```

`SimpleRNN` has `return_sequences=True` because the `LSTM` above it needs a sequence to read. The `LSTM` uses the default `False` because `Dense` wants one vector per review.

**About `model.summary()` in the next cell:** in current Keras versions it shows `0 (unbuilt)` because the model hasn't seen any data yet (`input_length` is deprecated and ignored). That's why there's a second `model.summary()` after `fit`, which shows the real shapes and the numbers above. Calling `model.build((None, 100))` before `summary()` also works.

In [ ]:
model = Sequential()

model.add(Embedding(20000, 128, input_length=100)) ## 2773 X 100 X 128 --> 3d
model.add(SimpleRNN(50,return_sequences=True)) ## 2773 X 100 X 50
model.add(LSTM(100))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))


# try using different optimizers and different optimizer configs
model.compile('adam', 'binary_crossentropy', metrics=['accuracy'])

model.summary()

### Explanation: What `fit` does with `batch_size=32`

2,773 reviews / 32 per batch = **87 steps** per epoch (the last batch is smaller). At each step: 32 reviews go through the whole shape walk above, the loss compares 32 probabilities with 32 labels, and every weight (embedding table, RNN, LSTM, Dense) gets nudged once.

In [ ]:
print('Train...')
history=model.fit(x_train, y_train,
          batch_size=32,
          epochs=1,
          validation_data=[x_test, y_test])

In [ ]:
model.summary()

## Explanation: Inside a Bidirectional LSTM

`Bidirectional(LSTM(100))` is simply **two separate LSTMs** with their own weights:

**Forward LSTM** reads the sentence left to right: `movie -> was -> not -> good`
**Backward LSTM** reads it right to left: `good -> not -> was -> movie`

Each produces its final `h` (100 numbers), and Keras **concatenates** them: `100 + 100 = 200`.

```
input to Bidirectional   (32, 100, 128)
forward LSTM  -> (32, 100)
backward LSTM -> (32, 100)
concatenate   -> (32, 200)
```

Why it helps: when the forward LSTM reaches "not", it hasn't seen "good" yet. The backward LSTM has already read "good" by the time it gets to "not". Together, the model has context from both sides.

**BiLSTM parameter count** = `2 x 4 x units x (input_features + units + 1)`. Here input features = 128 (straight from the Embedding): `2 x 4 x 100 x (128 + 100 + 1) = 183,200`. The `Dense(1)` after it has `200 + 1 = 201`.

The demo builds a tiny BiLSTM, runs each direction by hand, and checks that concatenating them matches Keras.

In [ ]:
# Toy demo: BiLSTM = forward LSTM + backward LSTM, concatenated
def lstm_by_hand(x, K, U, bias, n):
    h, c = np.zeros(n), np.zeros(n)
    for t in range(len(x)):
        z = x[t] @ K + h @ U + bias
        i, f, g, o = sigmoid(z[:n]), sigmoid(z[n:2*n]), np.tanh(z[2*n:3*n]), sigmoid(z[3*n:])
        c = f * c + i * g
        h = o * np.tanh(c)
    return h

tf.random.set_seed(7)
bi = Bidirectional(LSTM(3))                          # merge_mode='concat' is the default
keras_bi = bi(toy_emb).numpy()                       # (2, 5, 4) -> (2, 6)
Kf, Uf, bf, Kb, Ub, bb = bi.get_weights()            # forward weights, then backward weights

x = toy_emb[0]
h_forward  = lstm_by_hand(x,       Kf, Uf, bf, 3)    # read t = 0,1,2,3,4
h_backward = lstm_by_hand(x[::-1], Kb, Ub, bb, 3)    # read t = 4,3,2,1,0

print("forward  h:", h_forward)
print("backward h:", h_backward)
print("concat     :", np.concatenate([h_forward, h_backward]))
print("Keras      :", keras_bi[0])
print("Output shape:", keras_bi.shape, " -> 3 forward + 3 backward = 6")
print("Match?", np.allclose(np.concatenate([h_forward, h_backward]), keras_bi[0], atol=1e-5))

One detail worth mentioning: with `return_sequences=True`, Keras flips the backward outputs back into normal order before concatenating, so position `t` in the output holds "forward memory up to t" next to "backward memory from the end down to t".

The next cell is the complete BiLSTM script. Note it loads **all 25,000** reviews (no `maxlen` in `load_data`) and truncates each to its last 100 words with `pad_sequences`, so each epoch is 25000 / 32 = **782 steps**.

In [ ]:
#importing libraries
# from __future__ import print_function
import numpy as np

from tensorflow.keras.preprocessing import sequence
from keras.models import Sequential
from keras.layers import Dense, Dropout, Embedding, LSTM, Bidirectional
from keras.datasets import imdb


max_features = 20000
# cut texts after this number of words
# (among top max_features most common words)
maxlen = 100
batch_size = 32

print('Loading data...')
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)
print(len(x_train), 'train sequences')
print(len(x_test), 'test sequences')

print('Pad sequences (samples x time)')
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)
print('x_train shape:', x_train.shape)
print('x_test shape:', x_test.shape)
y_train = np.array(y_train)
y_test = np.array(y_test)

model = Sequential()

model.add(Embedding(max_features, 128, input_length=maxlen))
model.add(Bidirectional(LSTM(100)))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))


# try using different optimizers and different optimizer configs
model.compile('adam', 'binary_crossentropy', metrics=['accuracy'])

print('Train...')
history=model.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=1,
          validation_data=[x_test, y_test])

In [ ]:
model.summary()

In [ ]:
print(history.history['loss'])
print(history.history['accuracy'])

---
## Cheat sheet: the three layers side by side

| | SimpleRNN | LSTM | BiLSTM |
|---|---|---|---|
| Memory it carries | `h` | `h` and `c` | `h` and `c`, in each direction |
| Reads the sentence | left to right | left to right | both ways at once |
| Output width | `units` | `units` | `2 x units` |
| Parameters | `u(d+u+1)` | `4u(d+u+1)` | `2 x 4u(d+u+1)` |
| Input shape | `(batch, steps, d)` | `(batch, steps, d)` | `(batch, steps, d)` |
| `return_sequences=True` | `(batch, steps, u)` | `(batch, steps, u)` | `(batch, steps, 2u)` |
| `return_sequences=False` | `(batch, u)` | `(batch, u)` | `(batch, 2u)` |

`u` = units, `d` = features per time step (embedding size, or 1 if you feed raw IDs).

## Quick check for students

1. A batch of 16 reviews, padded to 60 words, goes into `Embedding(5000, 32)`. What shape comes out?
2. That goes into `LSTM(64, return_sequences=True)`. Output shape?
3. Then `Bidirectional(LSTM(64))`. Output shape?
4. How many parameters does `LSTM(64)` have if its input has 32 features?
5. Why would post-padding hurt a model that only uses the last hidden state?

*Answers: (16, 60, 32); (16, 60, 64); (16, 128); 4 x 64 x (32 + 64 + 1) = 24,832; the real words end up far from the output, with many zero steps diluting the memory in between.*